# Notebook 01 — Data Wrangling
**Capstone Project CC26-PRU469**: Skill Gap Simulator Dengan Career Scenario Engine

---

## Tujuan Notebook
1. Memuat semua raw data (job postings, job skills, skill master, user profiles, user skills)
2. Melakukan **assessing data** — cek missing values, duplikat, tipe data, outlier
3. Melakukan **cleaning data** secara manual
4. Menghasilkan dataset bersih di folder `data/processed/`

## Sumber Data
- **Base referensi**: Kaggle — *1.3M LinkedIn Jobs & Skills (2024)* by asaniczka
- **Metode**: Data sintetis realistis yang dibangun berdasarkan pola skill frequency dari dataset LinkedIn
- **Jumlah**: 600 job postings, 60 skill master, 750 user profiles

## 1. Import Library

In [17]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Library berhasil dimuat.')

Library berhasil dimuat.


## 2. Load Raw Data

In [18]:
# Path ke folder raw data
RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Load semua dataset
df_postings = pd.read_csv(os.path.join(RAW_DIR, 'job_postings_raw.csv'))
df_job_skills = pd.read_csv(os.path.join(RAW_DIR, 'job_skills_raw.csv'))
df_skill_master = pd.read_csv(os.path.join(RAW_DIR, 'skill_master_raw.csv'))
df_users = pd.read_csv(os.path.join(RAW_DIR, 'user_profiles_raw.csv'))
df_user_skills = pd.read_csv(os.path.join(RAW_DIR, 'user_skills_raw.csv'))

print(f'Job Postings     : {df_postings.shape}')
print(f'Job Skills       : {df_job_skills.shape}')
print(f'Skill Master     : {df_skill_master.shape}')
print(f'User Profiles    : {df_users.shape}')
print(f'User Skills      : {df_user_skills.shape}')

Job Postings     : (627, 9)
Job Skills       : (7144, 4)
Skill Master     : (61, 8)
User Profiles    : (772, 12)
User Skills      : (15450, 6)


## 3. Assessing Data

### 3.1 Job Postings

In [19]:
print('=== INFO ===')
print(df_postings.info())
print('\n=== HEAD ===')
df_postings.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 627 entries, 0 to 626
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   job_id         595 non-null    str  
 1   job_title      627 non-null    str  
 2   company        627 non-null    str  
 3   job_location   580 non-null    str  
 4   role_category  627 non-null    str  
 5   job_level      574 non-null    str  
 6   job_type       577 non-null    str  
 7   first_seen     577 non-null    str  
 8   source         627 non-null    str  
dtypes: str(9)
memory usage: 44.2 KB
None

=== HEAD ===


,job_id,job_title,company,job_location,role_category,job_level,job_type,first_seen,source
0,JOB-00001,Analytics Analyst,Tokopedia,"Jakarta, Indonesia",Data Analyst,Associate,Freelance,2024-05-20,Kaggle-LinkedIn-Synthetic
1,JOB-00002,Business Data Analyst,Gojek,"Yogyakarta, Indonesia",Data Analyst,Associate,Full-time,2024/12/12,Kaggle-LinkedIn-Synthetic
2,JOB-00003,BI Analyst,OCBC,"Bandung, Indonesia",Data Analyst,NaN,Part-time,2024-01-16,Kaggle-LinkedIn-Synthetic
3,JOB-00004,Business Data Analyst,Bank Mandiri,"Sydney, Australia",Data Analyst,Associate,Full-time,2024-11-04,Kaggle-LinkedIn-Synthetic
4,JOB-00005,Lead Data Analyst,Bukalapak,NaN,Data Analyst,Director,Internship,2024/12/25,Kaggle-LinkedIn-Synthetic


In [20]:
print('=== MISSING VALUES ===')
print(df_postings.isnull().sum())
print(f'\nPersentase missing per kolom:')
print((df_postings.isnull().sum() / len(df_postings) * 100).round(2))

=== MISSING VALUES ===
job_id           32
job_title         0
company           0
job_location     47
role_category     0
job_level        53
job_type         50
first_seen       50
source            0
dtype: int64

Persentase missing per kolom:
job_id          5.1000
job_title       0.0000
company         0.0000
job_location    7.5000
role_category   0.0000
job_level       8.4500
job_type        7.9700
first_seen      7.9700
source          0.0000
dtype: float64


In [21]:
print('=== DUPLIKAT ===')
print(f'Jumlah baris duplikat: {df_postings.duplicated().sum()}')
print(f'Jumlah job_id duplikat: {df_postings["job_id"].duplicated().sum()}')

=== DUPLIKAT ===
Jumlah baris duplikat: 18
Jumlah job_id duplikat: 57


In [22]:
print('=== STATISTIK DESKRIPTIF (Kategorikal) ===')
for col in ['role_category', 'job_level', 'job_type', 'source']:
    print(f'\n--- {col} ---')
    print(df_postings[col].value_counts())

=== STATISTIK DESKRIPTIF (Kategorikal) ===

--- role_category ---
role_category
Frontend Developer      202
Data Analyst            200
UI/UX Designer          194
UI/UX DESIGNER            6
DATA ANALYST              5
 Data Analyst             5
 FRONTEND DEVELOPER       3
 ui/ux designer           3
 data analyst             2
data analyst              1
FRONTEND DEVELOPER        1
 Frontend Developer       1
 frontend developer       1
 UI/UX DESIGNER           1
ui/ux designer            1
 UI/UX Designer           1
Name: count, dtype: int64

--- job_level ---
job_level
Associate           159
Mid-Senior level    152
Entry level         131
Director             60
Executive            32
mid level            10
Intern                8
Associate             7
Seniorr               4
director              3
entry-level           3
executive             3
Mid-Senior            2
Name: count, dtype: int64

--- job_type ---
job_type
Full-time     344
Internship     78
Contract       6

### 3.2 Job Skills

In [23]:
print('=== INFO ===')
print(df_job_skills.info())
print('\n=== HEAD ===')
df_job_skills.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 7144 entries, 0 to 7143
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   job_id          7144 non-null   str  
 1   skill_name      6857 non-null   str  
 2   skill_category  6864 non-null   str  
 3   role_category   7144 non-null   str  
dtypes: str(4)
memory usage: 223.4 KB
None

=== HEAD ===


,job_id,skill_name,skill_category,role_category
0,JOB-00001,Analytical Thinking,Soft Skill,Data Analyst
1,JOB-00001,Pandas,Tool,Data Analyst
2,JOB-00001,Statistics,Technical,Data Analyst
3,JOB-00001,Python,Technical,Data Analyst
4,JOB-00001,problem solving,Soft Skill,Data Analyst


In [24]:
print('=== MISSING VALUES ===')
print(df_job_skills.isnull().sum())
print(f'\n=== DUPLIKAT ===')
print(f'Baris duplikat: {df_job_skills.duplicated().sum()}')
print(f'\n=== UNIQUE SKILLS PER ROLE ===')
for role in df_job_skills['role_category'].unique():
    n = df_job_skills[df_job_skills['role_category'] == role]['skill_name'].nunique()
    print(f'  {role}: {n} unique skills')

=== MISSING VALUES ===
job_id              0
skill_name        287
skill_category    280
role_category       0
dtype: int64

=== DUPLIKAT ===
Baris duplikat: 227

=== UNIQUE SKILLS PER ROLE ===
  Data Analyst: 82 unique skills
  Frontend Developer: 81 unique skills
  UI/UX Designer: 91 unique skills


### 3.3 Skill Master

In [25]:
print('=== INFO ===')
print(df_skill_master.info())
print('\n=== HEAD ===')
df_skill_master.head(10)

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   role                       61 non-null     str    
 1   skill_name                 61 non-null     str    
 2   skill_category             61 non-null     str    
 3   importance_score           57 non-null     float64
 4   avg_learning_hours         59 non-null     float64
 5   market_demand_normal       61 non-null     float64
 6   market_demand_competitive  61 non-null     float64
 7   market_demand_booming      61 non-null     float64
dtypes: float64(5), str(3)
memory usage: 3.9 KB
None

=== HEAD ===


,role,skill_name,skill_category,importance_score,avg_learning_hours,market_demand_normal,market_demand_competitive,market_demand_booming
0,Data Analyst,Python,Technical,0.9200,200.0000,0.9200,1.0000,0.8300
1,Data Analyst,SQL,Technical,0.9500,120.0000,0.9500,1.0000,0.8500
2,Data Analyst,Statistics,Technical,0.8500,180.0000,0.8500,0.9800,0.7700
3,Data Analyst,Data Modeling,Technical,0.7000,150.0000,0.7000,0.8000,0.6300
4,Data Analyst,ETL Process,Technical,0.6500,100.0000,0.6500,0.7500,0.5900
5,Data Analyst,HYPOTHESIS TESTING,Technical,0.6000,80.0000,0.6000,0.6900,0.5400
6,Data Analyst,Regression Analysis,Technical,0.5800,90.0000,0.5800,0.6700,0.5200
7,Data Analyst,A/B Testing,Technical,0.5500,60.0000,0.5500,0.6300,0.5000
8,Data Analyst,Excel,Tool,0.8800,80.0000,0.8800,1.0000,0.7900
9,Data Analyst,Tableau,Tool,0.7800,100.0000,0.7800,0.9000,0.7000


In [26]:
print('=== MISSING VALUES ===')
print(df_skill_master.isnull().sum())
print(f'\n=== STATISTIK NUMERIK ===')
df_skill_master.describe()

=== MISSING VALUES ===
role                         0
skill_name                   0
skill_category               0
importance_score             4
avg_learning_hours           2
market_demand_normal         0
market_demand_competitive    0
market_demand_booming        0
dtype: int64

=== STATISTIK NUMERIK ===


,importance_score,avg_learning_hours,market_demand_normal,market_demand_competitive,market_demand_booming
count,57.0000,59.0000,61.0000,61.0000,61.0000
mean,0.7685,80.1695,0.7562,0.8557,0.6815
std,0.1409,45.3812,0.1445,0.1515,0.1291
min,0.4000,20.0000,0.4000,0.4600,0.3600
25%,0.6800,50.0000,0.6500,0.7500,0.5900
50%,0.8000,80.0000,0.8000,0.9200,0.7200
75%,0.8800,100.0000,0.8800,1.0000,0.7900
max,0.9750,250.0000,0.9800,1.0000,0.8800


### 3.4 User Profiles

In [27]:
print('=== INFO ===')
print(df_users.info())
print('\n=== HEAD ===')
df_users.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 772 entries, 0 to 771
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                772 non-null    str    
 1   target_role            772 non-null    str    
 2   background_level       696 non-null    str    
 3   study_hours_per_week   711 non-null    float64
 4   market_scenario        705 non-null    str    
 5   gap_score              736 non-null    float64
 6   readiness_label        772 non-null    str    
 7   top_missing_skill_1    772 non-null    str    
 8   top_missing_skill_2    772 non-null    str    
 9   top_missing_skill_3    772 non-null    str    
 10  estimated_weeks_ready  744 non-null    float64
 11  current_skills_json    772 non-null    str    
dtypes: float64(3), str(9)
memory usage: 72.5 KB
None

=== HEAD ===


,user_id,target_role,background_level,study_hours_per_week,market_scenario,gap_score,readiness_label,top_missing_skill_1,top_missing_skill_2,top_missing_skill_3,estimated_weeks_ready,current_skills_json
0,USR-0001,Data Analyst,Pemula,6.4000,Booming,0.7460,Significant Gap,Tableau,SQL,Analytical Thinking,130.7000,{bad_json
1,USR-0002,Data Analyst,NaN,5.1000,NaN,0.4415,Needs Work,Python,Excel,NumPy,126.1000,{bad_json
2,USR-0003,Data Analyst,Menengah,13.3000,Normal,0.3960,Almost Ready,SQL,Excel,Pandas,43.4000,"{""Python"": 59, ""SQL"": 39, ""Statistics"": 54, ""D..."
3,USR-0004,Data Analyst,Menengah,8.7000,Booming,0.4294,Needs Work,NumPy,ETL Process,Excel,54.4000,"{""Python"": 59, ""SQL"": 59, ""Statistics"": 55, ""D..."
4,USR-0005,Data Analyst,Pemula,19.0000,Kompetitif,0.7112,Significant Gap,Analytical Thinking,SQL,Excel,69.1000,"{""Python"": 46, ""SQL"": 12, ""Statistics"": 26, ""D..."


In [28]:
print('=== MISSING VALUES ===')
print(df_users.isnull().sum())
print(f'\n=== DUPLIKAT ===')
print(f'Baris duplikat: {df_users.duplicated().sum()}')
print(f'\n=== STATISTIK NUMERIK ===')
df_users[['study_hours_per_week', 'gap_score', 'estimated_weeks_ready']].describe()

=== MISSING VALUES ===
user_id                   0
target_role               0
background_level         76
study_hours_per_week     61
market_scenario          67
gap_score                36
readiness_label           0
top_missing_skill_1       0
top_missing_skill_2       0
top_missing_skill_3       0
estimated_weeks_ready    28
current_skills_json       0
dtype: int64

=== DUPLIKAT ===
Baris duplikat: 22

=== STATISTIK NUMERIK ===


,study_hours_per_week,gap_score,estimated_weeks_ready
count,711.0000,736.0000,744.0000
mean,10.5287,0.4909,75.2133
std,8.8241,0.2107,56.2474
min,-5.0000,0.1369,9.7000
25%,6.3000,0.3725,41.2750
50%,9.2000,0.4487,60.9000
75%,13.0000,0.7146,85.8250
max,80.0000,0.8176,466.6000


In [29]:
print('=== DISTRIBUSI KATEGORIKAL ===')
for col in ['target_role', 'background_level', 'market_scenario', 'readiness_label']:
    print(f'\n--- {col} ---')
    print(df_users[col].value_counts())

=== DISTRIBUSI KATEGORIKAL ===

--- target_role ---
target_role
Frontend Developer    260
Data Analyst          256
UI/UX Designer        256
Name: count, dtype: int64

--- background_level ---
background_level
Pemula       257
Menengah     237
Lanjutan     158
Lanjut        13
pemula        13
Beginner      10
Menengah       8
Name: count, dtype: int64

--- market_scenario ---
market_scenario
Booming        239
Normal         217
Kompetitif     207
normal          16
competitive     15
Kompetitif      11
Name: count, dtype: int64

--- readiness_label ---
readiness_label
Significant Gap    284
Needs Work         220
Almost Ready       127
Ready              104
unknown             34
Major Gap            3
Name: count, dtype: int64


### 3.5 User Skills

In [30]:
print('=== INFO ===')
print(df_user_skills.info())
print('\n=== HEAD ===')
df_user_skills.head()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 15450 entries, 0 to 15449
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   user_id            14808 non-null  str  
 1   target_role        15450 non-null  str  
 2   skill_name         14879 non-null  str  
 3   proficiency_level  15450 non-null  int64
 4   background_level   15450 non-null  str  
 5   market_scenario    15450 non-null  str  
dtypes: int64(1), str(5)
memory usage: 724.3 KB
None

=== HEAD ===


,user_id,target_role,skill_name,proficiency_level,background_level,market_scenario
0,USR-0001,Data Analyst,Python,30,Pemula,Booming
1,USR-0001,Data Analyst,SQL,29,Pemula,Booming
2,USR-0001,Data Analyst,Statistics,40,Pemula,Booming
3,USR-0001,Data Analyst,Data Modeling,16,Pemula,Booming
4,USR-0001,Data Analyst,ETL Process,17,Pemula,Booming


In [31]:
print('=== MISSING VALUES ===')
print(df_user_skills.isnull().sum())
print(f'\n=== STATISTIK PROFICIENCY ===')
df_user_skills['proficiency_level'].describe()

=== MISSING VALUES ===
user_id              642
target_role            0
skill_name           571
proficiency_level      0
background_level       0
market_scenario        0
dtype: int64

=== STATISTIK PROFICIENCY ===


count   15450.0000
mean       52.2534
std        28.9506
min       -10.0000
25%        31.0000
50%        50.0000
75%        73.0000
max       150.0000
Name: proficiency_level, dtype: float64

### 3.6 Ringkasan Temuan Assessing Data

| Dataset | Rows | Missing | Duplikat | Catatan |
|---|---|---|---|---|
| job_postings | 600 | Perlu dicek | Perlu dicek | Validasi tipe data |
| job_skills | 6,936 | Perlu dicek | Perlu dicek | Cek konsistensi skill name |
| skill_master | 60 | Perlu dicek | Perlu dicek | Cek range importance |
| user_profiles | 750 | Perlu dicek | Perlu dicek | Cek outlier gap_score |
| user_skills | 15,000 | Perlu dicek | Perlu dicek | Cek range proficiency |

## 4. Data Cleaning

### 4.1 Cleaning Job Postings

In [ ]:
import re

# Mapping Dictionaries untuk Standardisasi
ROLE_MAP = {
    "data analyst": "Data Analyst",
    "frontend developer": "Frontend Developer",
    "ui/ux designer": "UI/UX Designer",
    "ux designer": "UI/UX Designer",
    "ui designer": "UI/UX Designer",
}

JOB_LEVEL_MAP = {
    "entry level": "Entry level",
    "entry-level": "Entry level",
    "associate": "Associate",
    "mid level": "Mid-Senior level",
    "mid-senior": "Mid-Senior level",
    "mid-senior level": "Mid-Senior level",
    "senior": "Mid-Senior level",
    "seniorr": "Mid-Senior level",
    "director": "Director",
    "executive": "Executive",
}

JOB_TYPE_MAP = {
    "full time": "Full-time",
    "full-time": "Full-time",
    "part time": "Part-time",
    "part-time": "Part-time",
    "contract": "Contract",
    "contractor": "Contract",
    "intern": "Internship",
    "internship": "Internship",
    "freelance": "Freelance",
}

BACKGROUND_MAP = {
    "pemula": "Pemula",
    "beginner": "Pemula",
    "menengah": "Menengah",
    "lanjut": "Lanjutan",
    "lanjutan": "Lanjutan",
}

SCENARIO_MAP = {
    "normal": "Normal",
    "kompetitif": "Kompetitif",
    "competitive": "Kompetitif",
    "booming": "Booming",
}

SKILL_CATEGORY_MAP = {
    "technical": "Technical",
    "tool": "Tool",
    "soft skill": "Soft Skill",
    "softskill": "Soft Skill",
}

SKILL_ALIAS_MAP = {
    "java script": "JavaScript",
    "javasript": "JavaScript",
    "js": "JavaScript",
    "reactjs": "React",
    "pyhton": "Python",
    "python": "Python",
    "sql": "SQL",
    "s q l": "SQL",
    "powerbi": "Power BI",
    "power-bi": "Power BI",
    "ms excel": "Excel",
    "ms-excel": "Excel",
    "excell": "Excel",
    "a/btesting": "A/B Testing",
    "ab testing": "A/B Testing",
    "machine-learning": "Machine Learning",
    "ml": "Machine Learning",
    "userresearch": "User Research",
    "user-research": "User Research",
    "designsystem": "Design Systems",
    "design-systems": "Design Systems",
}

ACRONYM_SKILLS = {"SQL", "HTML", "CSS", "ETL", "UI", "UX", "AI"}

# Helper Functions untuk Normalisasi
def _normalize_text(value):
    if not isinstance(value, str):
        return value
    text = value.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" ,", ",")
    return text

def _smart_title(value):
    if not isinstance(value, str):
        return value
    text = _normalize_text(value)
    if text.isupper() and len(text) <= 5:
        return text
    text = text.title()
    text = text.replace("Ui/Ux", "UI/UX")
    return text

def _normalize_role(value):
    if not isinstance(value, str):
        return None
    key = _normalize_text(value).lower()
    return ROLE_MAP.get(key)

def _normalize_job_level(value):
    if not isinstance(value, str):
        return None
    key = _normalize_text(value).lower()
    return JOB_LEVEL_MAP.get(key)

def _normalize_job_type(value):
    if not isinstance(value, str):
        return None
    key = _normalize_text(value).lower()
    return JOB_TYPE_MAP.get(key)

def _normalize_background(value):
    if not isinstance(value, str):
        return None
    key = _normalize_text(value).lower()
    return BACKGROUND_MAP.get(key)

def _normalize_scenario(value):
    if not isinstance(value, str):
        return None
    key = _normalize_text(value).lower()
    return SCENARIO_MAP.get(key)

def _normalize_skill_category(value):
    if not isinstance(value, str):
        return None
    key = _normalize_text(value).lower()
    return SKILL_CATEGORY_MAP.get(key)

def _normalize_skill_name(value):
    if not isinstance(value, str):
        return None
    text = _normalize_text(value)
    key = text.lower().replace("-", " ").replace("/", " ")
    key = re.sub(r"\s+", " ", key)
    if key in SKILL_ALIAS_MAP:
        return SKILL_ALIAS_MAP[key]
    words = [w.upper() if w.upper() in ACRONYM_SKILLS else w.title() for w in key.split(" ")]
    return " ".join(words)

# --- CLEANING JOB POSTINGS ---
df_postings_clean = df_postings.copy()

# 1. Normalisasi text & kolom kategori
df_postings_clean["job_title"] = df_postings_clean["job_title"].apply(_smart_title)
df_postings_clean["company"] = df_postings_clean["company"].apply(_smart_title)
df_postings_clean["job_location"] = df_postings_clean["job_location"].apply(_smart_title)
df_postings_clean["role_category"] = df_postings_clean["role_category"].apply(_normalize_role)
df_postings_clean["job_level"] = df_postings_clean["job_level"].apply(_normalize_job_level)
df_postings_clean["job_type"] = df_postings_clean["job_type"].apply(_normalize_job_type)

# 2. Imputasi Missing Values
df_postings_clean["job_title"] = df_postings_clean["job_title"].fillna("Unknown Title")
mode_level = df_postings_clean["job_level"].mode(dropna=True)
mode_type = df_postings_clean["job_type"].mode(dropna=True)
df_postings_clean["job_level"] = df_postings_clean["job_level"].fillna(mode_level.iloc[0] if not mode_level.empty else "Mid-Senior level")
df_postings_clean["job_type"] = df_postings_clean["job_type"].fillna(mode_type.iloc[0] if not mode_type.empty else "Full-time")
df_postings_clean["role_category"] = df_postings_clean["role_category"].fillna("Data Analyst")

# 3. Tanggal (datetime conversion & imputation)
df_postings_clean["first_seen"] = pd.to_datetime(df_postings_clean["first_seen"], errors="coerce")
median_date = df_postings_clean["first_seen"].dropna().median()
df_postings_clean["first_seen"] = df_postings_clean["first_seen"].fillna(median_date)

# 4. Hapus Duplikat
before = len(df_postings_clean)
df_postings_clean = df_postings_clean.drop_duplicates(subset=["job_id", "job_title", "company", "first_seen"])
after = len(df_postings_clean)
print(f"Duplikat baris dihapus: {before - after}")

# 5. Penanganan Missing / Duplicate job_id
df_postings_clean["job_id"] = df_postings_clean["job_id"].fillna("")
df_postings_clean["job_id"] = df_postings_clean["job_id"].astype(str).str.strip()
existing_ids = df_postings_clean["job_id"].tolist()
next_id = 1

def _assign_id(current):
    global next_id
    if current and current != "nan" and current != "":
        return current
    while True:
        new_id = f"JOB-{next_id:05d}"
        next_id += 1
        if new_id not in existing_ids:
            existing_ids.append(new_id)
            return new_id

df_postings_clean["job_id"] = df_postings_clean["job_id"].apply(_assign_id)

# 6. Kolom Temporal
df_postings_clean["month"] = df_postings_clean["first_seen"].dt.month.astype(int)
df_postings_clean["quarter"] = df_postings_clean["first_seen"].dt.quarter.astype(int)

# Konversi datetime kembali ke format string untuk penyimpanan
df_postings_clean["first_seen"] = df_postings_clean["first_seen"].dt.strftime("%Y-%m-%d")

print(f"Shape setelah cleaning: {df_postings_clean.shape}")
df_postings_clean.dtypes


### 4.2 Cleaning Job Skills

In [ ]:
df_job_skills_clean = df_job_skills.copy()

# 1. Normalisasi kolom text
df_job_skills_clean["job_id"] = df_job_skills_clean["job_id"].astype(str).str.strip()
df_job_skills_clean["skill_name"] = df_job_skills_clean["skill_name"].apply(_normalize_skill_name)
df_job_skills_clean["skill_category"] = df_job_skills_clean["skill_category"].apply(_normalize_skill_category)
df_job_skills_clean["role_category"] = df_job_skills_clean["role_category"].apply(_normalize_role)

# 2. Hapus rows dengan key fields missing
df_job_skills_clean = df_job_skills_clean.dropna(subset=["job_id", "skill_name"])

# 3. Imputasi role_category dari df_postings_clean
role_from_job = df_postings_clean.set_index("job_id")["role_category"].to_dict()
df_job_skills_clean["role_category"] = df_job_skills_clean.apply(
    lambda r: r["role_category"] if pd.notna(r["role_category"]) else role_from_job.get(r["job_id"]), axis=1
)
df_job_skills_clean = df_job_skills_clean.dropna(subset=["role_category"])

# 4. Filter skill berdasarkan master list
temp_master = df_skill_master.copy()
temp_master["role"] = temp_master["role"].apply(_normalize_role)
temp_master["skill_name"] = temp_master["skill_name"].apply(_normalize_skill_name)
valid_pairs = set(zip(temp_master["role"], temp_master["skill_name"]))

df_job_skills_clean = df_job_skills_clean[df_job_skills_clean.apply(
    lambda r: (r["role_category"], r["skill_name"]) in valid_pairs, axis=1
)]

# 5. Overwrite skill_category dari master
master_cat = temp_master.set_index(["role", "skill_name"])["skill_category"].apply(_normalize_skill_category).to_dict()
df_job_skills_clean["skill_category"] = df_job_skills_clean.apply(
    lambda r: master_cat.get((r["role_category"], r["skill_name"]), r["skill_category"]), axis=1
)

# 6. Hapus duplikat
before = len(df_job_skills_clean)
df_job_skills_clean = df_job_skills_clean.drop_duplicates(subset=["job_id", "skill_name"])
after = len(df_job_skills_clean)
print(f"Duplikat dihapus: {before - after}")

# 7. Hitung frekuensi skill per role
skill_freq = df_job_skills_clean.groupby(["role_category", "skill_name"]).size().reset_index(name="frequency")
total_per_role = df_postings_clean.groupby("role_category").size().reset_index(name="total_postings")
skill_freq = skill_freq.merge(total_per_role, on="role_category")
skill_freq["frequency_pct"] = (skill_freq["frequency"] / skill_freq["total_postings"] * 100).round(2)

print(f"\nShape setelah cleaning: {df_job_skills_clean.shape}")
print(f"\nTop 5 skills per role (by frequency):")
for role in skill_freq["role_category"].unique():
    subset = skill_freq[skill_freq["role_category"] == role].nlargest(5, "frequency_pct")
    print(f"\n--- {role} ---")
    print(subset[["skill_name", "frequency", "frequency_pct"]].to_string(index=False))


### 4.3 Cleaning Skill Master

In [ ]:
df_skill_master_clean = df_skill_master.copy()

# 1. Normalisasi text
df_skill_master_clean["role"] = df_skill_master_clean["role"].apply(_normalize_role)
df_skill_master_clean["skill_name"] = df_skill_master_clean["skill_name"].apply(_normalize_skill_name)
df_skill_master_clean["skill_category"] = df_skill_master_clean["skill_category"].apply(_normalize_skill_category)

# 2. Validasi range numerik (clip ke 0-1)
for col in ["importance_score", "market_demand_normal", "market_demand_competitive", "market_demand_booming"]:
    df_skill_master_clean[col] = pd.to_numeric(df_skill_master_clean[col], errors="coerce").clip(0, 1)

# 3. Validasi learning hours (clip ke 20-400)
df_skill_master_clean["avg_learning_hours"] = pd.to_numeric(df_skill_master_clean["avg_learning_hours"], errors="coerce").clip(20, 400)

# 4. Hapus rows jika key fields missing
df_skill_master_clean = df_skill_master_clean.dropna(subset=["role", "skill_name", "skill_category"])

# 5. Imputasi missing values dengan median group-by
df_skill_master_clean["importance_score"] = df_skill_master_clean.groupby(["role"])["importance_score"].transform(
    lambda s: s.fillna(s.median())
)
df_skill_master_clean["avg_learning_hours"] = df_skill_master_clean.groupby(["skill_category"])["avg_learning_hours"].transform(
    lambda s: s.fillna(s.median())
)
for col in ["market_demand_normal", "market_demand_competitive", "market_demand_booming"]:
    df_skill_master_clean[col] = df_skill_master_clean.groupby(["role"])[col].transform(lambda s: s.fillna(s.median()))

# 6. Hapus duplikat
before = len(df_skill_master_clean)
df_skill_master_clean = df_skill_master_clean.drop_duplicates(subset=["role", "skill_name"])
after = len(df_skill_master_clean)

print(f"Duplikat dihapus: {before - after}")
print(f"importance_score range: [{df_skill_master_clean['importance_score'].min()}, {df_skill_master_clean['importance_score'].max()}]")
print(f"avg_learning_hours range: [{df_skill_master_clean['avg_learning_hours'].min()}, {df_skill_master_clean['avg_learning_hours'].max()}]")
print(f"Shape setelah cleaning: {df_skill_master_clean.shape}")


### 4.4 Cleaning User Profiles

In [ ]:
df_users_clean = df_users.copy()

# 1. Normalisasi text
df_users_clean["user_id"] = df_users_clean["user_id"].astype(str).str.strip()
df_users_clean["target_role"] = df_users_clean["target_role"].apply(_normalize_role)
df_users_clean["background_level"] = df_users_clean["background_level"].apply(_normalize_background)
df_users_clean["market_scenario"] = df_users_clean["market_scenario"].apply(_normalize_scenario)

# 2. Validasi study_hours_per_week (clip ke 3-25)
df_users_clean["study_hours_per_week"] = pd.to_numeric(df_users_clean["study_hours_per_week"], errors="coerce").clip(3, 25)

# 3. Hapus rows jika user_id missing
df_users_clean = df_users_clean.dropna(subset=["user_id"])

# 4. Imputasi kategori missing menggunakan mode / default
for col, default in [("target_role", "Data Analyst"), ("background_level", "Pemula"), ("market_scenario", "Normal")]:
    mode_value = df_users_clean[col].mode(dropna=True)
    df_users_clean[col] = df_users_clean[col].fillna(mode_value.iloc[0] if not mode_value.empty else default)

df_users_clean["study_hours_per_week"] = df_users_clean["study_hours_per_week"].fillna(df_users_clean["study_hours_per_week"].median())

# 5. Hapus duplikat
before = len(df_users_clean)
df_users_clean = df_users_clean.drop_duplicates(subset="user_id")
print(f"Duplikat dihapus: {before - len(df_users_clean)}")

# 6. Pembuatan Peta Skill dan Re-kalkulasi Metrik (Menyelaraskan dengan df_user_skills_clean)
def _build_role_skill_maps(df_sm):
    r_skills, r_importance, r_hours = {}, {}, {}
    for role in df_sm["role"].unique():
        subset = df_sm[df_sm["role"] == role]
        r_skills[role] = subset["skill_name"].tolist()
        r_importance[role] = dict(zip(subset["skill_name"], subset["importance_score"]))
        r_hours[role] = dict(zip(subset["skill_name"], subset["avg_learning_hours"]))
    return r_skills, r_importance, r_hours

role_skills, role_importance, role_hours = _build_role_skill_maps(df_skill_master_clean)

# Bersihkan user_skills temporer untuk re-kalkulasi
temp_user_skills = df_user_skills.copy()
temp_user_skills["user_id"] = temp_user_skills["user_id"].astype(str).str.strip()
temp_user_skills["target_role"] = temp_user_skills["target_role"].apply(_normalize_role)
temp_user_skills["skill_name"] = temp_user_skills["skill_name"].apply(_normalize_skill_name)
def _clean_proficiency(val):
    if isinstance(val, str) and val.endswith("%"):
        val = val.replace("%", "")
    return pd.to_numeric(val, errors="coerce")
temp_user_skills["proficiency_level"] = temp_user_skills["proficiency_level"].apply(_clean_proficiency).clip(0, 100)
temp_user_skills = temp_user_skills.dropna(subset=["user_id", "skill_name", "proficiency_level"])

skill_map = {}
for _, row in temp_user_skills.iterrows():
    skill_map.setdefault(row["user_id"], {})[row["skill_name"]] = row["proficiency_level"]

# Fungsi re-kalkulasi
def _compute_gap_score(prof, role, role_imp, scenario):
    mult = {"Normal": 1.0, "Kompetitif": 1.15, "Booming": 0.90}.get(scenario, 1.0)
    w_sum, imp_sum = 0, 0
    for sk, imp in role_imp[role].items():
        user_lvl = prof.get(sk, 0) / 100.0
        imp = min(1.0, imp * mult)
        w_sum += user_lvl * imp
        imp_sum += imp
    return round(1 - (w_sum / imp_sum), 4) if imp_sum > 0 else 1.0

def _find_top_missing_skills(prof, role, role_imp, n=3):
    g = []
    for sk, imp in role_imp[role].items():
        user_lvl = prof.get(sk, 0) / 100.0
        gap = imp * (1 - user_lvl)
        g.append((sk, gap))
    g.sort(key=lambda x: x[1], reverse=True)
    return [x[0] for x in g[:n]]

def _estimate_weeks_ready(prof, role, role_imp, role_hrs, study_hrs, scenario):
    mult = {"Normal": 1.0, "Kompetitif": 1.3, "Booming": 0.8}.get(scenario, 1.0)
    tot_hrs = 0
    for sk, imp in role_imp[role].items():
        user_lvl = prof.get(sk, 0) / 100.0
        gap = max(0, 1 - user_lvl)
        tot_hrs += gap * role_hrs[role].get(sk, 0) * imp
    return round((tot_hrs / study_hrs) * mult, 1) if study_hrs > 0 else 999

def _readiness_label(gap):
    if gap <= 0.2: return "Ready"
    if gap <= 0.4: return "Almost Ready"
    if gap <= 0.6: return "Needs Work"
    if gap <= 0.8: return "Significant Gap"
    return "Major Gap"

cleaned_records = []
for _, row in df_users_clean.iterrows():
    role = row["target_role"]
    if role not in role_skills:
        role = "Data Analyst"
    prof = {s: 0 for s in role_skills[role]}
    prof.update(skill_map.get(row["user_id"], {}))
    
    gap_score = _compute_gap_score(prof, role, role_importance, row["market_scenario"])
    top_missing = _find_top_missing_skills(prof, role, role_importance, n=3)
    est_weeks = _estimate_weeks_ready(prof, role, role_importance, role_hours, row["study_hours_per_week"], row["market_scenario"])
    readiness = _readiness_label(gap_score)
    
    cleaned_records.append({
        "user_id": row["user_id"],
        "target_role": role,
        "background_level": row["background_level"],
        "study_hours_per_week": round(float(row["study_hours_per_week"]), 1),
        "market_scenario": row["market_scenario"],
        "gap_score": gap_score,
        "readiness_label": readiness,
        "top_missing_skill_1": top_missing[0],
        "top_missing_skill_2": top_missing[1],
        "top_missing_skill_3": top_missing[2],
        "estimated_weeks_ready": est_weeks,
        "current_skills_json": json.dumps(prof),
    })

df_users_clean = pd.DataFrame(cleaned_records)
print(f"Shape setelah cleaning: {df_users_clean.shape}")


### 4.5 Cleaning User Skills

In [ ]:
df_user_skills_clean = df_user_skills.copy()

# 1. Normalisasi text
df_user_skills_clean["user_id"] = df_user_skills_clean["user_id"].astype(str).str.strip()
df_user_skills_clean["target_role"] = df_user_skills_clean["target_role"].apply(_normalize_role)
df_user_skills_clean["skill_name"] = df_user_skills_clean["skill_name"].apply(_normalize_skill_name)

# 2. Bersihkan proficiency level
def _clean_proficiency(value):
    if isinstance(value, str) and value.endswith("%"):
        value = value.replace("%", "")
    return pd.to_numeric(value, errors="coerce")

df_user_skills_clean["proficiency_level"] = df_user_skills_clean["proficiency_level"].apply(_clean_proficiency).clip(0, 100)

# 3. Hapus rows jika key fields missing
df_user_skills_clean = df_user_skills_clean.dropna(subset=["user_id", "skill_name", "proficiency_level"])

# 4. Imputasi target_role jika missing
if df_user_skills_clean["target_role"].isna().any():
    mode_role = df_user_skills_clean["target_role"].mode(dropna=True)
    df_user_skills_clean["target_role"] = df_user_skills_clean["target_role"].fillna(
        mode_role.iloc[0] if not mode_role.empty else "Data Analyst"
    )

# 5. Filter berdasarkan skill master
valid_pairs = set(zip(df_skill_master_clean["role"], df_skill_master_clean["skill_name"]))
df_user_skills_clean = df_user_skills_clean[df_user_skills_clean.apply(
    lambda r: (r["target_role"], r["skill_name"]) in valid_pairs, axis=1
)]

# 6. Hapus duplikat
before = len(df_user_skills_clean)
df_user_skills_clean = df_user_skills_clean.drop_duplicates(subset=["user_id", "skill_name"])
after = len(df_user_skills_clean)
print(f"Duplikat dihapus: {before - after}")
print(f"Shape setelah cleaning: {df_user_skills_clean.shape}")


## 5. Simpan Data Bersih

In [ ]:
# Simpan semua dataset bersih ke folder processed
df_postings_clean.to_csv(os.path.join(PROCESSED_DIR, 'job_postings_cleaned.csv'), index=False)
df_job_skills_clean.to_csv(os.path.join(PROCESSED_DIR, 'job_skills_cleaned.csv'), index=False)
df_skill_master_clean.to_csv(os.path.join(PROCESSED_DIR, 'skill_master_cleaned.csv'), index=False)
df_users_clean.to_csv(os.path.join(PROCESSED_DIR, 'user_profiles_cleaned.csv'), index=False)
df_user_skills_clean.to_csv(os.path.join(PROCESSED_DIR, 'user_skills_cleaned.csv'), index=False)

# Simpan juga skill frequency table
skill_freq.to_csv(os.path.join(PROCESSED_DIR, 'skill_frequency.csv'), index=False)

print('Semua dataset bersih telah disimpan ke data/processed/')
print('\nFile yang dihasilkan:')
for f in sorted(os.listdir(PROCESSED_DIR)):
    size = os.path.getsize(os.path.join(PROCESSED_DIR, f))
    print(f'  {f:40s} {size:>10,} bytes')


## 6. Validasi Akhir

In [ ]:
print('=== VALIDASI AKHIR ===')
print(f'\n1. Job Postings: {df_postings_clean.shape}')
print(f'   - Missing values: {df_postings_clean.isnull().sum().sum()}')
print(f'   - Duplikat: {df_postings_clean.duplicated().sum()}')

print(f'\n2. Job Skills: {df_job_skills_clean.shape}')
print(f'   - Missing values: {df_job_skills_clean.isnull().sum().sum()}')

print(f'\n3. Skill Master: {df_skill_master_clean.shape}')
print(f'   - Importance range: [{df_skill_master_clean["importance_score"].min()}, {df_skill_master_clean["importance_score"].max()}]')

print(f'\n4. User Profiles: {df_users_clean.shape}')
print(f'   - Gap score range: [{df_users_clean["gap_score"].min()}, {df_users_clean["gap_score"].max()}]')
print(f'   - Missing values: {df_users_clean.isnull().sum().sum()}')

print(f'\n5. User Skills: {df_user_skills_clean.shape}')
print(f'   - Proficiency range: [{df_user_skills_clean["proficiency_level"].min()}, {df_user_skills_clean["proficiency_level"].max()}]')
print(f'   - Missing values: {df_user_skills_clean.isnull().sum().sum()}')

print('\n=== DATA WRANGLING SELESAI ===')
